### Importing libraries

In [45]:
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from scipy import sparse

### loading the data

In [27]:
train = pd.read_parquet('../data/artifacts/03_train.parquet')

pd.set_option('display.max_columns', None)

train.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,item_count,total_price,total_freight_value,payment_count,total_payment,avg_product_name_length,avg_product_description_length,avg_product_height_cm,avg_product_length_cm,avg_product_photos_qty,avg_product_width_cm,avg_product_weight_g,product_count,seller_count,late
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,28013,1.0,58.900002,13.290000,1.0,72.190002,58.0,598.0,9.0,28.0,4.0,14.0,650.0,1.0,1.0,0
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,15775,1.0,239.899994,19.930000,1.0,259.829987,56.0,239.0,30.0,50.0,2.0,40.0,30000.0,1.0,1.0,0
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,35661,1.0,199.000000,17.870001,1.0,216.869995,59.0,695.0,13.0,33.0,2.0,33.0,3050.0,1.0,1.0,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,64b576fb70d441e8f1b2d7d446e483c5,varzea paulista,SP,13226,1.0,199.899994,18.139999,1.0,218.039993,59.0,409.0,40.0,35.0,1.0,30.0,3750.0,1.0,1.0,0
5,00048cc3ae777c65dbb7d2a0634bc1ea,816cbea969fe5b689b39cfc97a506742,delivered,2017-05-15 21:42:34,2017-05-17 03:55:27,2017-05-17 11:05:55,2017-05-22 13:44:35,2017-06-06,85c835d128beae5b4ce8602c491bf385,uberaba,MG,38017,1.0,21.900000,12.690000,1.0,34.590000,36.0,558.0,8.0,24.0,1.0,15.0,450.0,1.0,1.0,0


### Feature Engineering

In [28]:
def create_features(df):
    df = df.copy()

    df['purchase_year'] = df['order_purchase_timestamp'].dt.year
    df['purchase_month'] = df['order_purchase_timestamp'].dt.month
    df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
    df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour

    df['estimated_year'] = df['order_estimated_delivery_date'].dt.year

    df['estimated_month'] = df['order_estimated_delivery_date'].dt.month

    df['estimated_dayofweek'] = df['order_estimated_delivery_date'].dt.dayofweek


    # Drop some columns
    drop_cols = [
        'order_id',
        'customer_id',
        'customer_unique_id',
        'late',
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]

    df = df.drop(columns = drop_cols)

    return df


In [29]:
X_train = create_features(train)
y_train = train['late'].copy()

X_train.shape

(67533, 25)

In [30]:
X_train.dtypes

order_status                          str
customer_city                         str
customer_state                        str
customer_zip_code_prefix            int64
item_count                        float64
total_price                       float64
total_freight_value               float64
payment_count                     float64
total_payment                     float64
avg_product_name_length           float64
avg_product_description_length    float64
avg_product_height_cm             float64
avg_product_length_cm             float64
avg_product_photos_qty            float64
avg_product_width_cm              float64
avg_product_weight_g              float64
product_count                     float64
seller_count                      float64
purchase_year                       int32
purchase_month                      int32
purchase_dayofweek                  int32
purchase_hour                       int32
estimated_year                      int32
estimated_month                   

In [31]:
categorical_columns = [
    'order_status',
    'customer_city',
    'customer_state'
]

numerical_columns = [
    col for col in X_train.columns
    if col not in categorical_columns
]

numerical_columns

['customer_zip_code_prefix',
 'item_count',
 'total_price',
 'total_freight_value',
 'payment_count',
 'total_payment',
 'avg_product_name_length',
 'avg_product_description_length',
 'avg_product_height_cm',
 'avg_product_length_cm',
 'avg_product_photos_qty',
 'avg_product_width_cm',
 'avg_product_weight_g',
 'product_count',
 'seller_count',
 'purchase_year',
 'purchase_month',
 'purchase_dayofweek',
 'purchase_hour',
 'estimated_year',
 'estimated_month',
 'estimated_dayofweek']

In [32]:
# X_train['estimated_year'] = X_train['order_estimated_delivery_date'].dt.year

# X_train['estimated_month'] = X_train['order_estimated_delivery_date'].dt.month

# X_train['estimated_dayofweek'] = X_train['order_estimated_delivery_date'].dt.dayofweek

# X_train = X_train.drop(columns=['order_estimated_delivery_date'])

# X_train

In [33]:
# numerical_columns = [
#     col for col in X_train.columns
#     if col not in categorical_columns
# ]
# numerical_columns

In [34]:
numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median'))])

In [35]:
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown = 'ignore', sparse_output = True))
])

In [36]:
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numerical_columns),
    ('cat', categorical_transformer, categorical_columns)
])

In [37]:
X_train_transformed = preprocessor.fit_transform(X_train)

In [38]:
X_train.shape, X_train_transformed.shape

((67533, 25), (67533, 3798))

In [39]:
joblib.dump(preprocessor, '../data/artifacts/preprocessor.joblib')

['../data/artifacts/preprocessor.joblib']

In [40]:
feature_names = preprocessor.get_feature_names_out()

joblib.dump(feature_names, '../data/artifacts/feature_names.joblib')

['../data/artifacts/feature_names.joblib']

In [41]:
validation = pd.read_parquet('../data/artifacts/03_validation.parquet')
test = pd.read_parquet('../data/artifacts/03_test.parquet')


In [42]:
X_val = create_features(validation)
X_test = create_features(test)

y_val = validation.loc[:, 'late'].copy()
y_test = test.loc[:, 'late'].copy()

In [43]:
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)


In [44]:
print(X_train_transformed.shape)
print(X_val_transformed.shape)
print(X_test_transformed.shape)

(67533, 3798)
(14471, 3798)
(14472, 3798)


In [46]:
sparse.save_npz('../data/artifacts/X_train_features.npz', X_train_transformed)
sparse.save_npz('../data/artifacts/X_val_features.npz', X_val_transformed)
sparse.save_npz('../data/artifacts/X_test_features.npz', X_test_transformed)

y_train.to_csv('../data/artifacts/y_train.csv', index = False)
y_val.to_csv('../data/artifacts/y_val.csv', index = False)
y_test.to_csv('../data/artifacts/y_test.csv', index = False)


In [47]:
type(X_train_transformed)

scipy.sparse._csr.csr_matrix